# Tracer + ToyAgent -- Blackjack decision agent

This notebook walks through a **blackjack hit/stand decision agent** that contains one intentional silent bug, then audits it with Tracer's LLM judge.

The agent has three steps:
1. **`build_prompt`** -- compose the HIT/STAND prompt
2. **`call_llm`** -- send it to Groq (llama-3.3-70b) and store the raw response
3. **`parse_action`** -- extract the decision **(this is where the bug lives)**

The bug: `parse_action` scans for the *first* occurrence of `HIT` or `STAND` anywhere in the text instead of reading the dedicated `ACTION:` line. The LLM's chain-of-thought often says things like *'hitting here risks busting me'* before concluding `ACTION: STAND`, so the wrong word gets picked silently.

## 0. Setup

Make sure:
- `openai` is installed (`%pip install openai`)
- the Tracer repo exists at `~/Tracer`
- your `GROQ_API_KEY` and `OPENAI_API_KEY` are set below

In [ ]:
import os
import sys
from pathlib import Path

TRACER_DIR = Path.home() / 'Tracer'
assert TRACER_DIR.exists(), f'Tracer not found at {TRACER_DIR}'
if str(TRACER_DIR) not in sys.path:
    sys.path.insert(0, str(TRACER_DIR))

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from toy_agent import ToyAgent
from parser import parse_source
from executor import TracingExecutor
from judge import LLMJudge
from reporter import Reporter

print('Tracer dir:', TRACER_DIR)
print('GROQ_API_KEY set: ', bool(os.environ.get('GROQ_API_KEY')))
print('OPENAI_API_KEY set:', bool(os.environ.get('OPENAI_API_KEY')))

## 1. Set your API keys

- **`GROQ_API_KEY`** -- used by the agent to call the LLM
- **`OPENAI_API_KEY`** -- used by Tracer's judge (gpt-4o-mini) to evaluate each step

In [ ]:
import os

# Uncomment and fill in if not already set in your shell environment
# os.environ['GROQ_API_KEY']   = 'gsk-...'
# os.environ['OPENAI_API_KEY'] = 'sk-...'

groq_key   = os.environ.get('GROQ_API_KEY', '')
openai_key = os.environ.get('OPENAI_API_KEY', '')

if not groq_key:
    raise RuntimeError('Set GROQ_API_KEY before running the agent cells.')
if not openai_key:
    raise RuntimeError('Set OPENAI_API_KEY before running the Tracer audit cells.')

print('Both keys are set -- ready to go.')

## 2. Define the agent's goal

The **goal** is the single sentence Tracer's judge uses to evaluate every step. Be specific about what *correct* means -- vague goals produce vague verdicts.

In [ ]:
agent = ToyAgent(
    goal=(
        "Play one blackjack decision: call the LLM with the player's hand and dealer card, "
        "then parse the action from the final 'ACTION: HIT' or 'ACTION: STAND' line. "
        "state['action'] must match state['expected_action']."
    )
)

## 3. Register steps

### Imports and initial state

`agent.imports` are prepended to the generated script verbatim.

`agent.setup` must create a variable called `state` -- the dict threaded through every step. We use a single hand (16 vs dealer 7, correct answer = STAND) because Tracer tests one scenario at a time.

> **Why a dict?** Tracer wraps each step at definition time, so steps cannot close over module-level variables. Threading `state` through arguments sidesteps that. Give each step a docstring saying what it reads and writes -- the judge uses it as context.

In [ ]:
agent.imports = [
    'import os',
    'import re',
    'import openai',
]

# Hand: 16 vs dealer 7 -- correct basic-strategy answer is STAND
agent.setup = """
    state = {
        'hand_score':      16,
        'dealer_card':     '7',
        'strategy':        'cautious',
        'api_key':         os.environ.get('GROQ_API_KEY', ''),
        'prompt':          None,
        'llm_response':    None,
        'action':          None,
        'expected_action': 'STAND',
    }
"""
print('Setup defined.')

### Step 1 -- `build_prompt`

Composes the prompt that asks the LLM to think step by step and end with exactly one `ACTION:` line. This step is correct.

In [ ]:
@agent.add_step
def build_prompt(state):
    """Build a blackjack decision prompt.

    Reads state['strategy'], state['hand_score'], state['dealer_card'].
    Adds state['prompt']: instructs the LLM to end with exactly one line
    'ACTION: HIT' or 'ACTION: STAND'.
    """
    state['prompt'] = (
        f"You are a {state['strategy']} blackjack player.\n"
        f"Your hand totals {state['hand_score']}. Dealer shows {state['dealer_card']}.\n"
        f"Think step by step about whether to hit or stand.\n"
        f"End your response with exactly one line: ACTION: HIT  or  ACTION: STAND"
    )
    return state

### Step 2 -- `call_llm`

Sends the prompt to Groq (llama-3.3-70b-versatile via the OpenAI-compatible endpoint) and stores the raw text. This step is also correct.

In [ ]:
@agent.add_step
def call_llm(state):
    """Send the prompt to the LLM and store the raw text response.

    Reads state['prompt'] and state['api_key'].
    Adds state['llm_response']: the full text returned by the model.
    """
    import openai
    client = openai.OpenAI(
        api_key=state['api_key'],
        base_url='https://api.groq.com/openai/v1',
    )
    message = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        max_tokens=300,
        messages=[{'role': 'user', 'content': state['prompt']}],
    )
    state['llm_response'] = message.choices[0].message.content
    return state

### Step 3 -- `parse_action` (WARNING: the bug is here)

This step is supposed to read the dedicated `ACTION:` line at the end of the response. Instead the regex `(HIT|STAND)` matches the **first** occurrence of either word anywhere in the text.

The LLM's chain-of-thought frequently says things like *'hitting here risks busting'* before reaching its final `ACTION: STAND` -- so the wrong word gets returned silently, with no crash and no warning.

In [ ]:
@agent.add_step
def parse_action(state):
    """Parse the final ACTION line from the LLM response.

    Reads state['llm_response']; sets state['action'] to 'HIT' or 'STAND'
    by finding the line that begins with 'ACTION:' and extracting the word
    that follows. Falls back to 'STAND' if no ACTION line is present.
    state['action'] should equal state['expected_action'].
    """
    import re
    text = state['llm_response']
    # BUG: matches the FIRST occurrence of HIT/STAND anywhere in the text,
    # not the dedicated ACTION: line -- picks up chain-of-thought words like
    # 'hitting here would bust me' before the final ACTION: STAND.
    match = re.search(r'(HIT|STAND)', text)
    state['action'] = match.group(1) if match else 'STAND'
    return state

print('Registered steps:', [s.name for s in agent.steps])

## 4. Sanity-run the agent (no Tracer)

Run the agent locally first. The point is to show that it **finishes without crashing even when the answer is wrong** -- that silent success is exactly the problem Tracer is meant to catch.

In [ ]:
result = agent.run_local()

correct = result['action'] == result['expected_action']
print(f"Hand {result['hand_score']} vs dealer {result['dealer_card']}")
print(f"  Expected : {result['expected_action']}")
print(f"  Got      : {result['action']}  {'OK' if correct else '*** WRONG -- bug fired! ***'}")
print(f"\nReasoning snippet:")
print(result['llm_response'][:400])

## 5. See the bug across multiple hands

Running five different hands shows the bug fires **inconsistently** -- it depends on whether the LLM's reasoning happens to say `HIT` or `STAND` early in the chain of thought. This non-determinism makes it especially hard to catch without a systematic audit.

In [ ]:
import os

DEMO_HANDS = [
    {'hand_score': 16, 'dealer_card': '7',  'expected_action': 'STAND'},
    {'hand_score': 11, 'dealer_card': '10', 'expected_action': 'HIT'},
    {'hand_score': 19, 'dealer_card': 'A',  'expected_action': 'STAND'},
    {'hand_score': 15, 'dealer_card': '10', 'expected_action': 'HIT'},
    {'hand_score': 12, 'dealer_card': '4',  'expected_action': 'STAND'},
]

wrong = 0
for hand in DEMO_HANDS:
    initial = {
        **hand,
        'strategy':     'cautious',
        'api_key':      os.environ.get('GROQ_API_KEY', ''),
        'prompt':       None,
        'llm_response': None,
        'action':       None,
    }
    r = agent.run_local(initial=initial)
    correct = r['action'] == r['expected_action']
    if not correct:
        wrong += 1
    label = 'OK' if correct else '*** WRONG ***'
    print(f"Hand {r['hand_score']:>2} vs {r['dealer_card']:>2}  "
          f"expected={r['expected_action']:<5}  got={r['action']:<5}  {label}")

print(f"\n{wrong}/{len(DEMO_HANDS)} hands wrong")

## 6. See what Tracer will see

`agent.to_script()` stitches everything -- imports, initial state, step definitions, and a small runner -- into one Python source string. This is what Tracer's parser and executor consume.

In [ ]:
print(agent.to_script())

## 7. Audit with Tracer

One call: parse the script, execute step-by-step with the LLM judge attached, and print the report.

**What to watch for in the output:**
- `[OK]` -- judge was confident the step is correct
- `[!!]` -- judge flagged the step as incorrect -> will appear in Findings
- `[??]` -- judge returned UNKNOWN (not confident either way -> will **not** appear in Findings)

> **Note on silent errors:** If `parse_action` gets `[??]`, Findings will be empty even though the answer is wrong. The judge is an LLM that needs to understand blackjack strategy -- if it is not confident enough to say INCORRECT, nothing gets recorded. See the next section for the fix.

In [ ]:
result_audit = agent.audit_with_tracer(api_key=openai_key)

## 8. Read the findings

Each entry in `result_audit.errors` has a line number, error type, and the judge's explanation.

If the list is **empty** even though a hand was wrong, it means the judge returned `UNKNOWN` -- a second layer of silence on top of the first. The fix below forces a hard crash so the executor always catches it regardless of the judge's confidence.

In [ ]:
if result_audit.errors:
    print(f'Tracer found {len(result_audit.errors)} issue(s):\n')
    for err in result_audit.errors:
        print(f'  line {err.lineno:>3}  {err.error_type:<12}  {err.error_message}')
else:
    print('Findings: (empty)')
    print()
    print('The judge returned UNKNOWN -- not confident enough to flag parse_action.')
    print('Run the next section to add an assertion that forces a real crash.')

## 9. Fix 1 -- add an assertion to surface the bug

The structural gap is that there is no hard check that `state['action']` matches `state['expected_action']`. Adding `assert` at the end of `parse_action` turns the silent wrong answer into a real `AssertionError` -- no LLM judge confidence needed.

In [ ]:
asserted_agent = ToyAgent(goal=agent.goal)
asserted_agent.imports = agent.imports
asserted_agent.setup = agent.setup


@asserted_agent.add_step
def build_prompt(state):
    """Build a blackjack decision prompt (unchanged)."""
    state['prompt'] = (
        f"You are a {state['strategy']} blackjack player.\n"
        f"Your hand totals {state['hand_score']}. Dealer shows {state['dealer_card']}.\n"
        f"Think step by step about whether to hit or stand.\n"
        f"End your response with exactly one line: ACTION: HIT  or  ACTION: STAND"
    )
    return state


@asserted_agent.add_step
def call_llm(state):
    """Send the prompt to the LLM (unchanged)."""
    import openai
    client = openai.OpenAI(
        api_key=state['api_key'],
        base_url='https://api.groq.com/openai/v1',
    )
    message = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        max_tokens=300,
        messages=[{'role': 'user', 'content': state['prompt']}],
    )
    state['llm_response'] = message.choices[0].message.content
    return state


@asserted_agent.add_step
def parse_action(state):
    """Parse the ACTION line -- buggy regex, but now asserts correctness.

    Reads state['llm_response']; sets state['action'].
    Asserts state['action'] == state['expected_action'] so a wrong answer
    becomes a hard AssertionError instead of a silent mismatch.
    """
    import re
    text = state['llm_response']
    match = re.search(r'(HIT|STAND)', text)  # BUG still present
    state['action'] = match.group(1) if match else 'STAND'
    assert state['action'] == state['expected_action'], (
        f"Wrong action: got {state['action']!r}, expected {state['expected_action']!r}"
    )
    return state


print('Running asserted audit -- crash should now appear in Findings...')
result_asserted = asserted_agent.audit_with_tracer(api_key=openai_key, continue_on_error=True)
print()
print(f'Findings: {len(result_asserted.errors)} issue(s)')
for err in result_asserted.errors:
    print(f'  line {err.lineno:>3}  {err.error_type:<14}  {err.error_message[:80]}')

## 10. Fix 2 -- fix the actual bug (before/after)

The root cause is the regex. `re.search(r'(HIT|STAND)', text)` picks up the first word anywhere. The correct fix anchors to the `ACTION:` prefix:

```python
re.search(r'ACTION:\s*(HIT|STAND)', text)
```

This only matches the dedicated output line, ignoring all chain-of-thought text before it.

In [ ]:
fixed_agent = ToyAgent(goal=agent.goal)
fixed_agent.imports = agent.imports
fixed_agent.setup = agent.setup


@fixed_agent.add_step
def build_prompt(state):
    """Build a blackjack decision prompt (unchanged)."""
    state['prompt'] = (
        f"You are a {state['strategy']} blackjack player.\n"
        f"Your hand totals {state['hand_score']}. Dealer shows {state['dealer_card']}.\n"
        f"Think step by step about whether to hit or stand.\n"
        f"End your response with exactly one line: ACTION: HIT  or  ACTION: STAND"
    )
    return state


@fixed_agent.add_step
def call_llm(state):
    """Send the prompt to the LLM (unchanged)."""
    import openai
    client = openai.OpenAI(
        api_key=state['api_key'],
        base_url='https://api.groq.com/openai/v1',
    )
    message = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        max_tokens=300,
        messages=[{'role': 'user', 'content': state['prompt']}],
    )
    state['llm_response'] = message.choices[0].message.content
    return state


@fixed_agent.add_step
def parse_action(state):
    """Parse the ACTION line correctly.

    Reads state['llm_response']; sets state['action'] to 'HIT' or 'STAND'
    by anchoring to the 'ACTION:' prefix, ignoring all chain-of-thought text.
    state['action'] should equal state['expected_action'].
    """
    import re
    text = state['llm_response']
    # FIX: anchor to the ACTION: prefix so chain-of-thought words are ignored
    match = re.search(r'ACTION:\s*(HIT|STAND)', text)
    state['action'] = match.group(1) if match else 'STAND'
    return state


print('Fixed agent -- local run:')
r = fixed_agent.run_local()
print(f"  expected={r['expected_action']}  got={r['action']}  "
      f"{'OK' if r['action'] == r['expected_action'] else 'WRONG'}")

In [ ]:
print('Running Tracer on the fixed agent...')
result_fixed = fixed_agent.audit_with_tracer(api_key=openai_key)
print()
print('=== Before / After ===')
print(f'  Buggy agent  -- errors: {len(result_audit.errors)}')
print(f'  Fixed agent  -- errors: {len(result_fixed.errors)}')